In [5]:
from dotenv import load_dotenv
load_dotenv()

True

## Custom Middleware

> https://docs.langchain.com/oss/python/langchain/middleware/custom

### node-style

In [7]:
from dataclasses import dataclass

@dataclass
class Context:
    user_name: str
    age: int = 99

In [15]:
from langchain.agents.middleware import before_model

@before_model
def log_before_model(state, runtime):
    print(f"state: {state}")
    print(f"runtime: {runtime}")
    print(f"user_name: {runtime.context.user_name}")
    return None


In [17]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash", 
    tools=[],
    middleware=[log_before_model],
    context_schema=Context
)

In [38]:
agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐야?"}]},
    context=Context(user_name="김일남")
)

request: ModelRequest(model=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x000001836FBB5C70>, default_metadata=(), model_kwargs={}), messages=[HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='097d000c-a2ee-4704-8554-6fa1d1e3aa49')], system_message=None, tool_choice=None, tools=[], response_format=None, state={'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='097d000c-a2ee-4704-8554-6fa1d1e3aa49')]}, runtime=Runtime(context=Context(user_name='김일남', age=99

{'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='097d000c-a2ee-4704-8554-6fa1d1e3aa49'),
  AIMessage(content='김일남입니다.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c0eeb-593f-76b2-b403-055a469d14ea-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 52, 'total_tokens': 67, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 47}})]}

### wrap-style

In [ ]:
# from langchain.agents.middleware import wrap_model_call
# from langchain.messages import HumanMessage, SystemMessage

# @wrap_model_call
# def inject_user_name(request, handler):
#     print(f"request: {request}")
#     print("-" * 10)
#     return handler(request)


In [34]:
from langchain.agents.middleware import wrap_model_call
from langchain.messages import HumanMessage, SystemMessage

@wrap_model_call
def inject_user_name(request, handler):
    print(f"request: {request}")
    print("-" * 10)

    user_name =request.runtime.context.user_name
    
    if user_name:
        sys_prompt = f"사용자의 이름은 {user_name}입니다."
    else:
        sys_prompt = "사용자의 이름은 알려지지 않았습니다."
    request = request.override(system_prompt=sys_prompt)
    
    return handler(request)

In [35]:
from langchain.agents.middleware import after_model

@after_model
def log_after_model(state, runtime):
    print(f"after_model_state: {state}")
    print("-" * 10)
    return None

In [39]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash", 
    tools=[],
    middleware=[inject_user_name, log_after_model],
    context_schema=Context
)

In [40]:
agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐야?"}]},
    context=Context(user_name="김일남")
)

request: ModelRequest(model=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x000001836FE49C70>, default_metadata=(), model_kwargs={}), messages=[HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='f75e848e-aae7-41c4-ae38-39444a4125db')], system_message=None, tool_choice=None, tools=[], response_format=None, state={'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='f75e848e-aae7-41c4-ae38-39444a4125db')]}, runtime=Runtime(context=Context(user_name='김일남', age=99

{'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='f75e848e-aae7-41c4-ae38-39444a4125db'),
  AIMessage(content='김일남입니다.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c0eeb-94c8-7cf2-8edb-87ad6966a242-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 49, 'total_tokens': 64, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 44}})]}